## VPython could be used to produce the plots on the coordinate change for the paper on the Moon
You have to open it with jupyter, doesn't work on (my) VS code: jupyter notebook in terminal

This notebook should stay a stand-alone for illustration purposes only

In [ ]:
# Allows the modification of the files imported without having to restart kernel or re-import them
%load_ext autoreload
%autoreload 2

In [ ]:
import vpython as vpy
import numpy as np

# Parameters for drawing
opacity = 0.4        # between 0 (transparent) and 1 (opaque)
shininess = 0.8      # between 0 and 1
els_deg = 20       # elevation of the source, degrees
# elp_deg = 21.3       # elevation of the pointing, degrees
# azp_deg = 20       # azimuth of the pointing, degrees
N = 200              # nb of points on circle
radius_points = 0.02  # radius of the various points
dist_order1 = np.radians(10) # angular distance between order 0 and 1

el_source = np.radians(els_deg)
# el_pointing = np.radians(elp_deg)
# az_pointing = np.radians(azp_deg)

# Create scene
scene = vpy.canvas(title="Horizontal line of beam when pointing at source", background=vpy.color.white)

# Sphere of radius 1 centred at origin
vpy.sphere(pos=vpy.vec(0, 0, 0), radius=1, color=vpy.color.cyan,
       opacity=opacity, shininess=shininess)

# It seems that y is the vertical axis here, hence (x, y, z) becoming (y, z, x)

# S is the position of the source
S = vpy.vec(0, np.sin(el_source), np.cos(el_source))
vpy.sphere(pos=S, radius=radius_points, color=vpy.color.green)

# We plot the position of the beam order 1 peaks when the telesceop moints at the source
color = vpy.color.blue

def get_order1_pos(order0_pos, azel=False): # if not azel then cartesian vec
    # remember that the y axis is the vertical axis
    # also, order0_pos is a vec object from vpython if not azel
    if azel:
        order0_az, order0_el = order0_pos
    else:
        order0_az = np.arctan(order0_pos.x/order0_pos.z)
        order0_el = np.pi/2 - np.arccos(order0_pos.y)
    lon_zen = np.radians(np.arange(0, 360, 45))
    colat_zen = np.array([np.sqrt(2)*dist_order1, dist_order1]*4)
    crota1 = np.sin(order0_el) # cos rotation angle 90 - els_deg
    srota1 = np.cos(order0_el) # sin rotation angle 90 - els_deg
    #print("crota1 = {}, srota1 = {}".format(crota1, srota1))
    mat_rota1 = np.array([[     1,       0,      0],
                          [     0, crota1, -srota1],
                          [     0, srota1,  crota1]])
    crota2 = np.cos(order0_az)
    srota2 = np.sin(order0_az)
    #print("crota2 = {}, srota2 = {}".format(crota2, srota2))
    mat_rota2 = np.array([[ crota2,      0, srota2],
                          [      0,      1,      0],
                          [-srota2,      0, crota2]])
    rot_mats = [mat_rota1, mat_rota2]
    order1_pos = np.array([np.sin(lon_zen)*np.sin(colat_zen), np.cos(colat_zen), np.cos(lon_zen)*np.sin(colat_zen)]) # at zenith
    #shape (3, npoints)
    for rot_mat in rot_mats:
        order1_pos = np.einsum("ij,jk->ik", rot_mat, order1_pos) # we do the rotations in the order rota1 then rota2
    return order1_pos # cartesian always


order1_pos = get_order1_pos([0, el_source], azel=True)
#lon_zen = np.radians(np.arange(0, 360, 45))
#colat_zen = np.array([np.sqrt(2)*dist_order1, dist_order1]*4)
#crota = np.sin(el_source) # cos rotation angle 90 - els_deg
#srota = np.cos(el_source) # sin rotation angle 90 - els_deg
#pos_order1s = []
for i_order1 in range(8):
    #lono = lon_zen[i_order1]
    #colato = colat_zen[i_order1]
    #yo = np.cos(colato)
    #zo = np.cos(lono)*np.sin(colato)
    #xo = np.sin(lono)*np.sin(colato)
    #pos_order1s.append([xo, crota*yo - srota*zo, srota*yo + crota*zo])
    #pos = vpy.vec(xo, crota*yo - srota*zo, srota*yo + crota*zo)
    pos = vpy.vec(order1_pos[0, i_order1], order1_pos[1, i_order1], order1_pos[2, i_order1])
    vpy.sphere(pos=pos, radius=radius_points, color=color)

# P is the pointing direction of the telescope
# note that (cos(el), sin(el)) <--> (sin(theta), cos(theta)) for spherical coordinates
# P = vpy.vec(np.sin(az_pointing)*np.cos(el_pointing), np.sin(el_pointing), np.cos(az_pointing)*np.cos(el_pointing))
# here we choose Pto point at one of the positions of an order 1 peak of the beam centered on the source
i_order1_P = 2
P = vpy.vec(order1_pos[0, i_order1_P], order1_pos[1, i_order1_P], order1_pos[2, i_order1_P]) # right order 1
vpy.sphere(pos=P, radius=radius_points, color=vpy.color.red)

color = vpy.color.purple
order1_pos_ = get_order1_pos(P, azel=False)
for i_order1_ in range(8):
    pos = vpy.vec(order1_pos_[0, i_order1_], order1_pos_[1, i_order1_], order1_pos_[2, i_order1_])
    vpy.sphere(pos=pos, radius=radius_points, color=color)
    
# azimuth of pointing
azp = np.arctan(P.x/P.z) # arctan(y/x) becomes arctan(x/z) because y is vertical

# Great circle simulating the horizontal line of the beam of the telescope
# u = vpy.vec(1, 0, 0)
u = vpy.vec(np.cos(azp), 0, -np.sin(azp))
# vpy.sphere(pos=u, radius=0.04, color=vpy.color.blue)

cercle = vpy.curve(color=vpy.color.red, radius=0.003)
for i in range(N + 1):
    t = 2 * np.pi * i / N
    cercle.append(np.cos(t) * P + np.sin(t) * u)

print(order1_pos)

# Repère (optionnel)
vpy.arrow(pos=vpy.vec(0, 0, 0), axis=vpy.vec(1.5, 0, 0), color=vpy.color.red, shaftwidth=0.01)
vpy.arrow(pos=vpy.vec(0, 0, 0), axis=vpy.vec(0, 1.5, 0), color=vpy.color.green, shaftwidth=0.01)
vpy.arrow(pos=vpy.vec(0, 0, 0), axis=vpy.vec(0, 0, 1.5), color=vpy.color.blue, shaftwidth=0.01)